# Setup do Ambiente — Judicial Analytics Platform

Este notebook configura **todo o ambiente do zero**. Deve ser rodado uma única vez por workspace.
Se o workspace for recriado ou resetado, rode este notebook antes de qualquer outro.

**Estado que este notebook reproduz:**
```
judicial/
├── bronze/
│   ├── Volumes/raw_files/
│   │   ├── TJSC/
│   │   ├── TJPR/
│   │   └── processados/TJSC/ e processados/TJPR/
│   └── Tables/ (criadas pelos notebooks de ingestão)
├── silver/
└── gold/
```

## 1. Verificar ambiente

In [ ]:
# Confirmar que o Spark está funcionando antes de qualquer coisa
print(f"Spark version: {spark.version}")

## 2. Criar Catalog

In [ ]:
%sql
-- Catalog raiz do projeto
-- Agrupa todos os schemas: bronze, silver, gold
CREATE CATALOG IF NOT EXISTS judicial

## 3. Criar Schemas (camadas medalhão)

In [ ]:
%sql
-- bronze: dados brutos da API, sem transformação
CREATE SCHEMA IF NOT EXISTS judicial.bronze

In [ ]:
%sql
-- silver: dados limpos, deduplicados e enriquecidos
CREATE SCHEMA IF NOT EXISTS judicial.silver

In [ ]:
%sql
-- gold: modelo analítico final (dimensões e fatos)
CREATE SCHEMA IF NOT EXISTS judicial.gold

## 4. Criar Volume de arquivos brutos

O Volume `raw_files` é onde os arquivos JSON gerados pelo M2 ficam
antes de serem lidos pelo notebook `bronze_databricks`.

Substituiu o DBFS (`dbfs:/FileStore/`) que não está disponível na Free Edition.

In [ ]:
%sql
-- Volume principal de arquivos brutos
-- Caminho resultante: /Volumes/judicial/bronze/raw_files/
CREATE VOLUME IF NOT EXISTS judicial.bronze.raw_files

## 5. Criar estrutura de diretórios no Volume

Cada tribunal tem sua própria pasta dentro do Volume.
A pasta `processados/` recebe os arquivos após serem ingeridos na Bronze
— evita que o mesmo arquivo seja lido duas vezes em caso de reexecução.

In [ ]:
VOLUME_BASE = "/Volumes/judicial/bronze/raw_files"
TRIBUNAIS   = ["TJSC", "TJPR"]

# Pastas de entrada — onde o M2 deposita os arquivos JSON
for tribunal in TRIBUNAIS:
    dbutils.fs.mkdirs(f"{VOLUME_BASE}/{tribunal}/")
    print(f"Criado: {VOLUME_BASE}/{tribunal}/")

# Pastas de arquivo — onde os arquivos vão após serem processados
for tribunal in TRIBUNAIS:
    dbutils.fs.mkdirs(f"{VOLUME_BASE}/processados/{tribunal}/")
    print(f"Criado: {VOLUME_BASE}/processados/{tribunal}/")

## 6. Verificação final

Rode todas as células abaixo e confirme que a saída bate com o esperado antes de avançar.

In [ ]:
%sql
-- Esperado: bronze, default, gold, information_schema, silver
SHOW SCHEMAS IN judicial

In [ ]:
%sql
-- Esperado: raw_files
SHOW VOLUMES IN judicial.bronze

In [ ]:
# Esperado: TJSC/, TJPR/, processados/
display(dbutils.fs.ls("/Volumes/judicial/bronze/raw_files/"))

In [ ]:
# Esperado: TJSC/, TJPR/ dentro de processados
display(dbutils.fs.ls("/Volumes/judicial/bronze/raw_files/processados/"))

## 7. Notas sobre o estado atual

### Volume `datajud` (não utilizado)
Durante o setup inicial foi criado um volume chamado `datajud` em `judicial.bronze`.
Ele não é usado pelo pipeline — o volume correto é `raw_files`.
Pode ser removido com:
```sql
DROP VOLUME IF EXISTS judicial.bronze.datajud
```

### Tabelas existentes
As tabelas Delta **não são criadas por este notebook** — elas são criadas
automaticamente pelos notebooks de ingestão quando rodam pela primeira vez:

| Tabela | Criada por |
|---|---|
| `judicial.bronze.datajud_raw` | `bronze_databricks.ipynb` |
| `judicial.silver.movimentos` | `silver_transform.ipynb` (M4) |
| `judicial.gold.dim_*` | modelos dbt (M5) |
| `judicial.gold.fato_movimentos` | modelos dbt (M6) |

### Checklist antes de avançar
```
✓ Spark version printando sem erro
✓ SHOW SCHEMAS retorna: bronze, silver, gold
✓ SHOW VOLUMES retorna: raw_files
✓ ls do volume mostra: TJSC/, TJPR/, processados/
✓ ls de processados/ mostra: TJSC/, TJPR/
```